In [3]:
import os
import pandas as pd
import json
import re
from itertools import product

# Liste aller Instanzen
instances = [
    "a3_o80_m10_an10_ar9_reduced",
    "a5_o96_m10_an10_ar10_reduced",
    "a10_o107_m5_an57_ar12",
    "a10_o114_m6_an57_ar11",
    "a10_o128_m6_an51_ar13",
    "a10_o144_m6_an53_ar12",
    "a15_o170_m9_an80_ar18",
    "a20_o236_m12_an106_ar24",
    "a25_o306_m13_an127_ar31",
    "a30_o355_m18_an148_ar42",
    "a40_o476_m22_an215_ar51",
    "a50_o578_m28_an276_ar66",
]

# Alle möglichen Ziele und Methoden
objectives = ["3 Objectives", "6 Objectives"]
methods = ["costs", "weighted", "hierarchical", "hierarchical_tolerance"]

# Generiere alle möglichen Kombinationen
all_combinations = list(product(instances, objectives, methods))

# Funktion zum Extrahieren von Zahlen für die richtige Sortierung
def sort_instances(instance_name):
    numbers = re.findall(r'\d+', instance_name)
    return tuple(int(num) for num in numbers)  # Zahlen extrahieren und als Tupel zurückgeben

# Funktion zum Generieren des Logdateinamens basierend auf Instanz und Strategie
def generate_log_filename(json_file_path, instance, strategy):
    log_filename = f"gurobi_{instance}_{strategy}.log"
    base_dir = os.path.dirname(json_file_path)
    return os.path.join(base_dir, log_filename)

# Funktion zum Extrahieren des MIP-Gaps und zum Überprüfen der Logs bei hierarchical_tolerance
def extract_mip_gap_and_check_log_hierarchical_tolerance(log_file_path_1, log_file_path_2):
    mip_gap = "-"
    star_index = None

    # Erster Logfile prüfen
    try:
        with open(log_file_path_1, 'r') as log_file:
            lines = log_file.readlines()
            if "Time Limit reached" in "".join(lines):
                for line in reversed(lines):
                    match = re.search(r'gap\s+([\d\.]+)%', line)
                    if match:
                        mip_gap = float(match.group(1))  # Gap als Prozentwert extrahieren
                        star_index = 1  # Construction Fulfillment
                        return round(mip_gap, 4), star_index
    except FileNotFoundError:
        pass

    # Zweiter Logfile prüfen
    try:
        with open(log_file_path_2, 'r') as log_file:
            lines = log_file.readlines()
            if "Time Limit reached" in "".join(lines):
                for line in reversed(lines):
                    match = re.search(r'gap\s+([\d\.]+)%', line)
                    if match:
                        mip_gap = float(match.group(1))  # Gap als Prozentwert extrahieren
                        break

                # Optimierungsschritt im zweiten Logfile prüfen
                for line in reversed(lines):
                    if "Multi-objectives: optimize objective" in line:
                        match = re.search(r'objective\s+(\d+)', line)
                        if match:
                            star_index = int(match.group(1)) + 1  # Offset um 1, da CF bereits behandelt wurde
                            break
    except FileNotFoundError:
        pass

    return round(mip_gap, 4) if isinstance(mip_gap, float) else mip_gap, star_index

# Funktion zum Extrahieren des MIP-Gaps und zum Überprüfen von "Time Limit reached"
def extract_mip_gap_and_check_log(log_file_path, strategy):
    mip_gap = "-"
    star_index = None
    try:
        with open(log_file_path, 'r') as log_file:
            lines = log_file.readlines()
            if "Time Limit reached" in "".join(lines):
                for line in reversed(lines):
                    match = re.search(r'gap\s+([\d\.]+)%', line)
                    if match:
                        mip_gap = float(match.group(1))  # Gap als Prozentwert extrahieren
                        break
                
                # Nur für "hierarchical" den Optimierungsschritt prüfen
                if strategy.startswith("hierarchical"):
                    for line in reversed(lines):
                        if "Multi-objectives: optimize objective" in line:
                            match = re.search(r'objective\s+(\d+)', line)
                            if match:
                                star_index = int(match.group(1))
                            break
    except FileNotFoundError:
        pass
    return round(mip_gap, 2) if isinstance(mip_gap, float) else mip_gap, star_index

# Basispfad (aktueller Ordner des Jupyter Notebooks)
solution_path = os.getcwd()

# Benutzerdefinierte Strategiereihenfolge
strategy_order = {"costs": 0, "weighted": 1, "hierarchical": 2, "hierarchical_tolerance": 3}

# Liste für die Daten
data_objectives = []

# Ordner rekursiv durchlaufen
for root, dirs, files in os.walk(solution_path):
    for file in files:
        if file.endswith(".json") and "Solution_Construction" in file:
            file_path = os.path.join(root, file)
            
            # JSON-Datei einlesen mit Fehlerbehandlung
            try:
                with open(file_path, 'r') as f:
                    data = json.load(f)
            except json.JSONDecodeError:
                continue
            
            # Instanzname, Objective-Type und Strategie aus dem Pfad extrahieren
            instance = os.path.basename(os.path.dirname(os.path.dirname(root)))
            objectives_type = os.path.basename(os.path.dirname(root)).replace("_", " ")
            strategy = os.path.basename(os.path.dirname(file_path))
            
            # Logdatei suchen
            log_filename_1 = generate_log_filename(file_path, instance, strategy)
            log_filename_2 = log_filename_1.replace(".log", "_round_2.log")
            
            # MIP-Gap und Stern-Index extrahieren
            if strategy == "hierarchical_tolerance":
                mip_gap, star_index = extract_mip_gap_and_check_log_hierarchical_tolerance(log_filename_1, log_filename_2)
            else:
                mip_gap, star_index = extract_mip_gap_and_check_log(log_filename_1, strategy)
            
            # Mapping der Objectives mit den Namen aus der anderen Tabelle
            result_dict = {
                "Instance": instance,
                "Number of Objectives": objectives_type,
                "Method": strategy,
                "Runtime": round(data.get("RechenzeitInSekunden", None), 2),
                "Construction Fulfillment": data.get("Baustellenfertig", None),
                "Driver Violation": data.get("NichtregulaereFahrer", None),
                "Commute Distance": round(data.get("ArbeitswegGesamt", None), 1),
                "Transport Distance": round(data.get("TransportdistanzGesamt", None), 1),
                "Machine Count": data.get("MaschinenGenutzt", None),
                "Worker Count": data.get("ArbeiterGenutzt", None),
                "Gap (%)": mip_gap,
            }
            
            # Stern bei hierarchischen Strategien einfügen
            if star_index and strategy.startswith("hierarchical"):
                objective_mapping = {
                    1: "Construction Fulfillment",
                    2: "Driver Violation",
                    3: "Commute Distance",
                    4: "Transport Distance",
                    5: "Machine Count",
                    6: "Worker Count"
                }
                key = objective_mapping.get(star_index)
                if key and result_dict[key] is not None:
                    result_dict[key] = f"{result_dict[key]}*"
            
            # Hinzufügen zur Liste
            data_objectives.append(result_dict)

# Daten in DataFrame umwandeln
df_objectives = pd.DataFrame(data_objectives)

# Strategiereihenfolge hinzufügen
df_objectives["Strategy_Order"] = df_objectives["Method"].map(strategy_order)

# Sortieren: Erst nach Instanz, dann nach Number of Objectives und Method
df_objectives["Instance_Sort"] = df_objectives["Instance"].map(sort_instances)
df_objectives = df_objectives.sort_values(by=["Instance_Sort", "Number of Objectives", "Strategy_Order"]).drop(columns=["Instance_Sort", "Strategy_Order"]).reset_index(drop=True)

# Fehlende Kombinationen prüfen und hinzufügen
existing_combinations = set(zip(df_objectives["Instance"], df_objectives["Number of Objectives"], df_objectives["Method"]))
missing_combinations = [comb for comb in all_combinations if comb not in existing_combinations]

# Fehlende Kombinationen mit `"-"` auffüllen
missing_rows = []
for instance, objective, method in missing_combinations:
    missing_rows.append({
        "Instance": instance,
        "Number of Objectives": objective,
        "Method": method,
        "Construction Fulfillment": "-",
        "Driver Violation": "-",
        "Commute Distance": "-",
        "Transport Distance": "-",
        "Machine Count": "-",
        "Worker Count": "-",
        "Runtime": "-",
        "Gap (%)": "-",
    })

# Fehlende Kombinationen zum DataFrame hinzufügen
df_objectives = pd.concat([df_objectives, pd.DataFrame(missing_rows)], ignore_index=True)

# Werte von hierarchical auf hierarchical_tolerance übertragen (inklusive Runtime)
def adjust_tolerance_from_hierarchical(df):
    # Iteriere über alle Reihen, die zur Methode hierarchical gehören
    for idx, row in df.iterrows():
        if row["Method"] == "hierarchical":
            instance = row["Instance"]
            objective = row["Number of Objectives"]

            # Prüfe, ob ein "*" in Construction Fulfillment oder Driver Violation enthalten ist
            if "*" in str(row["Construction Fulfillment"]) or "*" in str(row["Driver Violation"]):
                # Finde die entsprechende Zeile für hierarchical_tolerance
                tolerance_row_idx = df[
                    (df["Instance"] == instance) &
                    (df["Number of Objectives"] == objective) &
                    (df["Method"] == "hierarchical_tolerance")
                ].index

                if not tolerance_row_idx.empty:
                    # Übernehme die Werte von hierarchical (inklusive Runtime)
                    df.loc[tolerance_row_idx, [
                        "Construction Fulfillment",
                        "Driver Violation",
                        "Commute Distance",
                        "Transport Distance",
                        "Machine Count",
                        "Worker Count",
                        "Gap (%)",
                        "Runtime",  # Runtime hinzufügen
                    ]] = row[
                        [
                            "Construction Fulfillment",
                            "Driver Violation",
                            "Commute Distance",
                            "Transport Distance",
                            "Machine Count",
                            "Worker Count",
                            "Gap (%)",
                            "Runtime",  # Runtime übernehmen
                        ]
                    ].values
    return df

# Anwenden der Anpassungen
df_objectives = adjust_tolerance_from_hierarchical(df_objectives)

# Sortieren und anzeigen
df_objectives = df_objectives.sort_values(by=["Instance", "Number of Objectives", "Method"]).reset_index(drop=True)

# Werte formatieren
df_objectives["Commute Distance"] = df_objectives["Commute Distance"].map(lambda x: f"{x:.1f}" if isinstance(x, (int, float)) else x)
df_objectives["Transport Distance"] = df_objectives["Transport Distance"].map(lambda x: f"{x:.1f}" if isinstance(x, (int, float)) else x)
df_objectives["Runtime"] = df_objectives["Runtime"].map(lambda x: f"{x:.2f}" if isinstance(x, (int, float)) else x)
df_objectives["Gap (%)"] = df_objectives["Gap (%)"].map(lambda x: f"{x:.2f}" if isinstance(x, (int, float)) else x)

# Wenn Construction Fulfillment == 0, alle Objective-Werte und Gap auf "-" setzen (außer Runtime)
columns_to_update = [
    "Construction Fulfillment",
    "Driver Violation",
    "Commute Distance",
    "Transport Distance",
    "Machine Count",
    "Worker Count",
    "Gap (%)",
]

df_objectives.loc[df_objectives["Construction Fulfillment"] == 0, columns_to_update] = "-"
df_objectives.loc[df_objectives["Construction Fulfillment"] == "0*", columns_to_update] = "-"

# Strategiereihenfolge
strategy_order = {"costs": 0, "weighted": 1, "hierarchical": 2, "hierarchical_tolerance": 3}

# Reihenfolge der Objectives
objective_order = {"3 Objectives": 0, "6 Objectives": 1}

# Sortieren der Instanzen, Objectives und Methoden
df_objectives["Instance_Sort"] = df_objectives["Instance"].map(sort_instances)
df_objectives["Objective_Order"] = df_objectives["Number of Objectives"].map(objective_order)
df_objectives["Strategy_Order"] = df_objectives["Method"].map(strategy_order)

df_objectives = df_objectives.sort_values(
    by=["Instance_Sort", "Objective_Order", "Strategy_Order"]
).drop(columns=["Instance_Sort", "Objective_Order", "Strategy_Order"]).reset_index(drop=True)

# Tabelle anzeigen
df_objectives.style

,Instance,Number of Objectives,Method,Runtime,Construction Fulfillment,Driver Violation,Commute Distance,Transport Distance,Machine Count,Worker Count,Gap (%)
0,a3_o80_m10_an10_ar9_reduced,3 Objectives,costs,7.70,3,42,3620.9,1365.6,2,8,-
1,a3_o80_m10_an10_ar9_reduced,3 Objectives,weighted,19.48,3,38,3844.7,1447.4,2,8,-
2,a3_o80_m10_an10_ar9_reduced,3 Objectives,hierarchical,44.73,3,38,3844.7,1447.4,2,8,-
3,a3_o80_m10_an10_ar9_reduced,3 Objectives,hierarchical_tolerance,38.19,3,46,3473.5,1701.1,2,8,-
4,a3_o80_m10_an10_ar9_reduced,6 Objectives,costs,3080.95,3,40,4635.6,70.6,2,6,-
5,a3_o80_m10_an10_ar9_reduced,6 Objectives,weighted,808.78,2,8,2697.7,29.7,2,5,-
6,a3_o80_m10_an10_ar9_reduced,6 Objectives,hierarchical,1296.71,3,38,3844.7,917.4,2,8,-
7,a3_o80_m10_an10_ar9_reduced,6 Objectives,hierarchical_tolerance,273.29,3,46,3811.4,99.2,2,8,-
8,a5_o96_m10_an10_ar10_reduced,3 Objectives,costs,16.27,5,14,5148.0,191.7,7,10,-
9,a5_o96_m10_an10_ar10_reduced,3 Objectives,weighted,46.04,5,1,6109.3,191.7,7,9,-


# Analyse
### Übergreifend über Instanzen
- Gap Entwicklung bei weighted und costs für 6 Objectives (vllt auch 3)
- Optimization Step analyse bei hierarichal und tolerance --> wie viele steps wurde geschafft? Bei Gleichtand auch Gap mit einbeziehen für 6 Objectives (vllt auch 3)
- Frage: "Bis wann macht Solver überhaupt sinn?"

### Einzelne Instazen
- Analyse welche Methode die "beste" ist --> Vor un NAchteile der Methoden
    - z.B. keien Garantie für die meisten Baustellen bei weighted (oder costs?)
    - schnell keien Beachtung untergeordneter Ziele bei hierarichal und tolreance da nahc 3 h abbruch
    - bei cosst und wieghetd vorteil da auch schon vorher alle zeiele mit betrachtet werden udn d er gap sich auf alle bezieht
    - bei costs stammfahrer viel weniger gewicht als bei weighetd dadruch höhere anzahl (erkennbar auch höhere rechnezeit?)
        - direkte gewichte (gradienten) vergleichen
    - tolerance auch höhere stammfahrer da tolerance immer ausgenutzt wird 

In [4]:
df_filtered = df_objectives[df_objectives["Number of Objectives"] == "6 Objectives"]
df_filtered = df_filtered[df_filtered["Method"] == "costs"]

# Tabelle anzeigen
df_filtered.style

,Instance,Number of Objectives,Method,Runtime,Construction Fulfillment,Driver Violation,Commute Distance,Transport Distance,Machine Count,Worker Count,Gap (%)
4,a3_o80_m10_an10_ar9_reduced,6 Objectives,costs,3080.95,3,40,4635.6,70.6,2,6,-
12,a5_o96_m10_an10_ar10_reduced,6 Objectives,costs,97.91,5,10,6432.3,95.9,4,8,-
20,a10_o107_m5_an57_ar12,6 Objectives,costs,73.58,9,22,3844.8,358.2,4,7,-
28,a10_o114_m6_an57_ar11,6 Objectives,costs,936.42,9,28,5031.1,266.3,5,8,-
36,a10_o128_m6_an51_ar13,6 Objectives,costs,10838.16,8,18,6619.0,449.2,4,6,0.06
44,a10_o144_m6_an53_ar12,6 Objectives,costs,124.18,6,11,4332.0,101.1,4,5,-
52,a15_o170_m9_an80_ar18,6 Objectives,costs,10810.27,15,64,13249.9,426.5,8,14,0.03
60,a20_o236_m12_an106_ar24,6 Objectives,costs,10801.46,16,83,8561.7,525.8,5,12,11.30
68,a25_o306_m13_an127_ar31,6 Objectives,costs,10802.06,18,47,6767.0,478.8,7,14,24.96
76,a30_o355_m18_an148_ar42,6 Objectives,costs,10801.48,20,37,9960.4,1048.4,6,13,49.85


In [5]:
# Filter
df_filtered = df_objectives[df_objectives["Number of Objectives"] == "6 Objectives"]
df_filtered = df_filtered[df_filtered["Method"] == "hierarchical"]
# Bis auf eine 10er Instanz alle 10er Instanzen entfernen
df_filtered = df_filtered[df_filtered["Instance"] != "a10_o114_m6_an57_ar11"]
df_filtered = df_filtered[df_filtered["Instance"] != "a10_o128_m6_an51_ar13"]
df_filtered = df_filtered[df_filtered["Instance"] != "a10_o144_m6_an53_ar12"]

# Tabelle anzeigen
df_filtered.style

,Instance,Number of Objectives,Method,Runtime,Construction Fulfillment,Driver Violation,Commute Distance,Transport Distance,Machine Count,Worker Count,Gap (%)
6,a3_o80_m10_an10_ar9_reduced,6 Objectives,hierarchical,1296.71,3,38,3844.7,917.4,2,8,-
14,a5_o96_m10_an10_ar10_reduced,6 Objectives,hierarchical,1242.01,5,1,6109.3,191.7,6,9,-
22,a10_o107_m5_an57_ar12,6 Objectives,hierarchical,9317.38,9,2,5018.2,642.7,5,9,-
54,a15_o170_m9_an80_ar18,6 Objectives,hierarchical,10801.54,15,34*,20084.1,5827.5,9,18,100.00
62,a20_o236_m12_an106_ar24,6 Objectives,hierarchical,10800.59,17,76*,16667.5,5546.0,12,19,100.00
70,a25_o306_m13_an127_ar31,6 Objectives,hierarchical,10800.96,22,141*,22226.9,9319.6,13,28,99.29
78,a30_o355_m18_an148_ar42,6 Objectives,hierarchical,10819.43,23*,355,19682.7,6787.1,17,39,30.43
86,a40_o476_m22_an215_ar51,6 Objectives,hierarchical,10803.06,26*,442,20200.7,8101.3,22,33,46.15
94,a50_o578_m28_an276_ar66,6 Objectives,hierarchical,10803.10,-,-,-,-,-,-,-


In [6]:
# Filter
df_filtered = df_objectives[df_objectives["Number of Objectives"] == "6 Objectives"]
df_filtered = df_filtered[df_filtered["Instance"] == "a10_o128_m6_an51_ar13"]

# Tabelle anzeigen
df_filtered.style

,Instance,Number of Objectives,Method,Runtime,Construction Fulfillment,Driver Violation,Commute Distance,Transport Distance,Machine Count,Worker Count,Gap (%)
36,a10_o128_m6_an51_ar13,6 Objectives,costs,10838.16,8,18,6619.0,449.2,4,6,0.06
37,a10_o128_m6_an51_ar13,6 Objectives,weighted,10806.35,8,0,6628.5,359.7,5,7,0.80
38,a10_o128_m6_an51_ar13,6 Objectives,hierarchical,4870.95,8,0,6057.5,673.4,6,7,-
39,a10_o128_m6_an51_ar13,6 Objectives,hierarchical_tolerance,10806.37,8,10,5478.7,183.8*,6,9,34.43


### Latex Vorbereitung

In [7]:
# Filter für Instanzen, die mit "a3", "a5" oder "a10" beginnen
df_filtered = df_objectives[df_objectives["Instance"].str.match(r"^(a3_|a5_|a10)")]
df_filtered = df_filtered[df_filtered["Number of Objectives"] == "6 Objectives"]


# 1. Simplify the Instance names (retain until the secound "_")
df_filtered['Instance'] = df_filtered['Instance'].str.extract(r'^([^_]*_[^_]*)')

# 2. Drop the 'Number of Objectives' column
df_filtered = df_filtered.drop(columns=['Number of Objectives'])

# 3. Shorten 'hierarchical_tolerance' in the 'Method' column to 'tolerance'
df_filtered['Method'] = df_filtered['Method'].replace({'hierarchical_tolerance': 'tolerance'})

# 4. Convert Runtime from seconds to minutes and round to whole numbers
df_filtered['Runtime'] = pd.to_numeric(df_filtered['Runtime'], errors='coerce')  # Convert to numeric
df_filtered['Runtime'] = (df_filtered['Runtime'] / 60).round(0).astype('Int64')  # Convert to minutes and round

# Round Gap (%) to one decimal place and keep "-" as is
df_filtered['Gap (%)'] = df_filtered['Gap (%)'].apply(
    lambda x: x if x == "-" else f"{float(x):.1f}" if pd.notna(x) else x
)

# Display the updated DataFrame
df_filtered.style


,Instance,Method,Runtime,Construction Fulfillment,Driver Violation,Commute Distance,Transport Distance,Machine Count,Worker Count,Gap (%)
4,a3_o80,costs,51,3,40,4635.6,70.6,2,6,-
5,a3_o80,weighted,13,2,8,2697.7,29.7,2,5,-
6,a3_o80,hierarchical,22,3,38,3844.7,917.4,2,8,-
7,a3_o80,tolerance,5,3,46,3811.4,99.2,2,8,-
12,a5_o96,costs,2,5,10,6432.3,95.9,4,8,-
13,a5_o96,weighted,4,5,1,6242.4,122.3,5,9,-
14,a5_o96,hierarchical,21,5,1,6109.3,191.7,6,9,-
15,a5_o96,tolerance,180,5,11,5706.3,26.4,6,9*,11.1
20,a10_o107,costs,1,9,22,3844.8,358.2,4,7,-
21,a10_o107,weighted,52,9,2,5632.6,296.2,5,8,-


In [8]:
# Filter für Instanzen, die mit "a3", "a5" oder "a10" beginnen
df_filtered = df_objectives[df_objectives["Instance"].str.match(r"^(a15|a2|a30|a40|a50)")]
df_filtered = df_filtered[df_filtered["Number of Objectives"] == "6 Objectives"]


# 1. Simplify the Instance names (retain until the secound "_")
df_filtered['Instance'] = df_filtered['Instance'].str.extract(r'^([^_]*_[^_]*)')

# 2. Drop the 'Number of Objectives' column
df_filtered = df_filtered.drop(columns=['Number of Objectives'])

# 3. Shorten 'hierarchical_tolerance' in the 'Method' column to 'tolerance'
df_filtered['Method'] = df_filtered['Method'].replace({'hierarchical_tolerance': 'tolerance'})

# 4. Convert Runtime from seconds to minutes and round to whole numbers
df_filtered['Runtime'] = pd.to_numeric(df_filtered['Runtime'], errors='coerce')  # Convert to numeric
df_filtered['Runtime'] = (df_filtered['Runtime'] / 60).round(0).astype('Int64')  # Convert to minutes and round

# Round Gap (%) to one decimal place and keep "-" as is
df_filtered['Gap (%)'] = df_filtered['Gap (%)'].apply(
    lambda x: x if x == "-" else f"{float(x):.1f}" if pd.notna(x) else x
)


# Display the updated DataFrame
df_filtered.style


,Instance,Method,Runtime,Construction Fulfillment,Driver Violation,Commute Distance,Transport Distance,Machine Count,Worker Count,Gap (%)
52,a15_o170,costs,180,15,64,13249.9,426.5,8,14,0.0
53,a15_o170,weighted,180,14,2,10412.4,413.9,8,13,11.3
54,a15_o170,hierarchical,180,15,34*,20084.1,5827.5,9,18,100.0
55,a15_o170,tolerance,180,15,34*,20084.1,5827.5,9,18,100.0
60,a20_o236,costs,180,16,83,8561.7,525.8,5,12,11.3
61,a20_o236,weighted,180,14,9,7354.8,529.5,6,11,37.8
62,a20_o236,hierarchical,180,17,76*,16667.5,5546.0,12,19,100.0
63,a20_o236,tolerance,180,17,76*,16667.5,5546.0,12,19,100.0
68,a25_o306,costs,180,18,47,6767.0,478.8,7,14,25.0
69,a25_o306,weighted,180,16,9,6061.4,428.3,7,14,47.9


In [15]:
# Filter für Instanzen, die mit "a3", "a5" oder "a10" beginnen
df_filtered = df_objectives[df_objectives["Instance"].str.match(r"^(a3|a10|a15|a2|a30|a40|a5)")]
df_filtered = df_filtered[df_filtered["Number of Objectives"] == "6 Objectives"]
df_filtered = df_filtered[df_filtered["Method"] == "hierarchical"]


# 1. Simplify the Instance names (retain until the secound "_")
df_filtered['Instance'] = df_filtered['Instance'].str.extract(r'^([^_]*_[^_]*)')

# 2. Drop the 'Number of Objectives' column
df_filtered = df_filtered.drop(columns=['Number of Objectives'])

# 3. Shorten 'hierarchical_tolerance' in the 'Method' column to 'tolerance'
#df_filtered['Method'] = df_filtered['Method'].replace({'hierarchical_tolerance': 'tolerance'})

# 4. Convert Runtime from seconds to minutes and round to whole numbers
df_filtered['Runtime'] = pd.to_numeric(df_filtered['Runtime'], errors='coerce')  # Convert to numeric
df_filtered['Runtime'] = (df_filtered['Runtime'] / 60).round(0).astype('Int64')  # Convert to minutes and round

# Round Gap (%) to one decimal place and keep "-" as is
df_filtered['Gap (%)'] = df_filtered['Gap (%)'].apply(
    lambda x: x if x == "-" else f"{float(x):.1f}" if pd.notna(x) else x
)

# Display the updated DataFrame
df_filtered.style


,Instance,Method,Runtime,Construction Fulfillment,Driver Violation,Commute Distance,Transport Distance,Machine Count,Worker Count,Gap (%)
6,a3_o80,hierarchical,22,3,38,3844.7,917.4,2,8,-
14,a5_o96,hierarchical,21,5,1,6109.3,191.7,6,9,-
22,a10_o107,hierarchical,155,9,2,5018.2,642.7,5,9,-
30,a10_o114,hierarchical,180,9,5,6831.5,1367.5*,6,8,89.2
38,a10_o128,hierarchical,81,8,0,6057.5,673.4,6,7,-
46,a10_o144,hierarchical,180,6,1,4448.4,373.2,6*,7,33.3
54,a15_o170,hierarchical,180,15,34*,20084.1,5827.5,9,18,100.0
62,a20_o236,hierarchical,180,17,76*,16667.5,5546.0,12,19,100.0
70,a25_o306,hierarchical,180,22,141*,22226.9,9319.6,13,28,99.3
78,a30_o355,hierarchical,180,23*,355,19682.7,6787.1,17,39,30.4
